# Day 2 · 한국어 회의 Workflow와 Agent

하나의 입력을 8개 차시 동안 확장합니다. 웹사이트 확인이 아니라 코드·명령·test·결과 파일을 직접 다루며, 외부 서비스 저장·게시·발송은 dry-run과 사람 승인을 먼저 거칩니다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, os, subprocess, sys

def find_workspace(start):
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-day1.txt").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("WORKSPACE_ROOT_NOT_FOUND")

ROOT = find_workspace(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": ROOT.name, "python": sys.version.split()[0]})

{'workspace': 'llm-agent-and-workflow-automation', 'python': '3.12.12'}


In [2]:
# Run All은 설치가 끝난 환경에서 network 호출 없이 실행합니다.
# 처음 한 번만 아래 flag를 True로 바꿔 현재 Notebook Kernel에 설치합니다.
INSTALL_CORE_DEPENDENCIES = False

dependency_groups = {
    "core": (["pydantic", "pytest", "langchain_core", "langgraph"], ROOT / "requirements-day1.txt"),
}
install_flags = {
    "core": INSTALL_CORE_DEPENDENCIES,
}
dependency_status = {}
for group, (modules, requirements_path) in dependency_groups.items():
    missing = [name for name in modules if importlib.util.find_spec(name) is None]
    if missing and install_flags[group]:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
            check=True,
        )
        missing = [name for name in modules if importlib.util.find_spec(name) is None]
    dependency_status[group] = {
        "ready": not missing,
        "missing": missing,
        "install_command": f"python -m pip install -r {requirements_path.relative_to(ROOT)}",
        "network_used_by_run_all": bool(install_flags[group]),
    }

assert dependency_status["core"]["ready"], (
    "CORE_DEPENDENCIES_MISSING: 위 INSTALL_CORE_DEPENDENCIES를 True로 바꾸고 이 셀만 먼저 실행하세요."
)
print(json.dumps(dependency_status, ensure_ascii=False, indent=2))

# faster-whisper model과 공개 음성 다운로드는 2차시 opt-in 셀에서만 실행합니다.

{
  "core": {
    "ready": true,
    "missing": [],
    "install_command": "python -m pip install -r requirements-day1.txt",
    "network_used_by_run_all": false
  }
}


In [3]:
REFERENCE_OUT = ROOT / "output/course-labs/day2-v2"
OUT = REFERENCE_OUT / "student-run"
OUT.mkdir(parents=True, exist_ok=True)

TRACK_RUN_MANIFEST = True
RUN_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
RUN_RESULT_FILES = []
RUN_TEST_EVIDENCE = []

def write_run_manifest():
    live_opt_ins = {
        "openai": os.getenv("OPENAI_LIVE_OPT_IN", "0") == "1",
        "ollama": os.getenv("OLLAMA_LIVE_OPT_IN", "0") == "1",
        "faster_whisper": os.getenv("FASTER_WHISPER_LIVE_OPT_IN", "0") == "1",
    }
    completed_periods = sorted({
        name.split("_", 1)[0]
        for name in RUN_RESULT_FILES
        if len(name) >= 3 and name[:2].isdigit() and name[2] == "_"
    })
    manifest = {
        "run_started_at_utc": RUN_STARTED_AT_UTC,
        "python_version": sys.version.split()[0],
        "reference_outputs_read_only": str(REFERENCE_OUT.relative_to(ROOT)),
        "student_run_directory": str(OUT.relative_to(ROOT)),
        "completed_periods": completed_periods,
        "result_files": [str((OUT / name).relative_to(ROOT)) for name in RUN_RESULT_FILES],
        "tests": RUN_TEST_EVIDENCE,
        "safety": {
            "default_lane_network_free": True,
            "this_run_network_free": not any(live_opt_ins.values()),
            "live_opt_ins": live_opt_ins,
            "external_write": False,
            "automatic_email_send": False,
        },
    }
    path = OUT / "run_manifest.json"
    path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return path

def record_test_evidence(name, result):
    evidence = {
        "name": name,
        "command": result["command"],
        "return_code": result["returncode"],
        "status": "PASS" if result["returncode"] == 0 else "FAIL",
    }
    RUN_TEST_EVIDENCE[:] = [
        item for item in RUN_TEST_EVIDENCE if item["name"] != name
    ]
    RUN_TEST_EVIDENCE.append(evidence)
    if TRACK_RUN_MANIFEST:
        write_run_manifest()
    return evidence

def save_json(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    if TRACK_RUN_MANIFEST:
        if name not in RUN_RESULT_FILES:
            RUN_RESULT_FILES.append(name)
        write_run_manifest()
    print({"saved": str(path.relative_to(ROOT))})
    return path

def save_text(name, text):
    path = OUT / name
    path.write_text(text.rstrip() + "\n", encoding="utf-8")
    if TRACK_RUN_MANIFEST:
        if name not in RUN_RESULT_FILES:
            RUN_RESULT_FILES.append(name)
        write_run_manifest()
    print({"saved": str(path.relative_to(ROOT))})
    return path

if TRACK_RUN_MANIFEST:
    write_run_manifest()

def run_command(*args, cwd=ROOT):
    completed = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    display_args = list(args)
    if display_args and display_args[0] == sys.executable:
        display_args[0] = "python"
    result = {
        "command": " ".join(display_args),
        "returncode": completed.returncode,
        "stdout_tail": completed.stdout.strip().splitlines()[-5:],
        "stderr_tail": completed.stderr.strip().splitlines()[-5:],
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

## 1차시 · Meeting Agent Architecture

일반 사용자의 말로 시작해도 구현에서는 입력·맥락·변환·검증·승인·초안을 분리합니다. LLM은 한 단계의 판단 도구이고, Agent는 요청에 따라 정보원과 Workflow를 고르는 상위 실행 구조입니다.

In [4]:
from src.course_services.day2_meeting_workflow import (
    DEFAULT_OPENAI_MODEL, DomainContext, MCPRetrievalPolicy, MeetingRecord,
    SourceInput, TranscriptEnvelope, build_mcp_retrieval_plan,
    build_interruptible_meeting_graph,
    compact_workflow_result, compare_execution_strategies,
    diagnose_provider_options, normalize_source, render_email_draft,
    external_action_approval_gate, run_optional_local_stt_smoke,
    resume_interruptible_meeting_review, route_execution_strategy,
    run_meeting_workflow, start_interruptible_meeting_review,
    run_optional_cli_prompt, run_optional_openai_prompt, run_optional_openai_record,
    source_mixing_error_example,
    validate_model_record_output, validate_record_evidence,
)

architecture = {
    "user_request": "회의를 이해하고 근거 있는 기록·할 일·인사이트 초안을 만들어 줘",
    "layers": [
        {"order": 1, "name": "policy", "question": "읽어도 되는 정보와 하면 안 되는 행동은?"},
        {"order": 2, "name": "input_adapter", "question": "Meet·ClovaNote·음성 중 어떤 한 입력인가?"},
        {"order": 3, "name": "domain_context", "question": "산업 용어와 이전 결정은 무엇인가?"},
        {"order": 4, "name": "workflow", "question": "정규화→STT→구조화→근거 검증 순서는?"},
        {"order": 5, "name": "human_review", "question": "누가 승인·수정·거절하는가?"},
        {"order": 6, "name": "draft_export", "question": "MD·이메일 초안을 어디까지 만들 것인가?"},
    ],
    "two_meanings_of_agent": {
        "user_view": "여러 단계를 알아서 이어 주는 서비스",
        "engineering_view": "상태·도구·정책·오류·승인을 가진 실행 시스템",
    },
    "fixed_graph": [
        "policy", "input_normalize", "stt_optional", "structure",
        "evidence", "human_review", "export_draft",
    ],
    "invariants": {
        "source_count": 1,
        "human_review_required": True,
        "external_write": False,
        "run_all_network_calls": 0,
    },
}
route_cases = {
    "single_llm": route_execution_strategy(
        requested_actions=["rewrite_as_podcast_script"]
    ),
    "deterministic_workflow": route_execution_strategy(
        requested_actions=["normalize", "summarize", "todos", "draft"]
    ),
    "agent_router": route_execution_strategy(
        requested_actions=["summarize", "todos"],
        external_context_sources=["notion", "slack"],
    ),
}
external_action_gate = {
    "without_human_approval": external_action_approval_gate(
        "send_meeting_email", human_approved=False
    ),
    "approved_dry_run": external_action_approval_gate(
        "send_meeting_email", human_approved=True
    ),
}
architecture["three_route_cases"] = route_cases
architecture["external_action_approval_gate"] = external_action_gate
assert architecture["invariants"]["external_write"] is False
assert {item["strategy"] for item in route_cases.values()} == {
    "single_llm", "deterministic_workflow", "agent_router",
}
assert external_action_gate["without_human_approval"]["error_code"] == (
    "EXTERNAL_ACTION_HUMAN_APPROVAL_REQUIRED"
)
assert external_action_gate["approved_dry_run"]["executed"] is False
save_json("01_architecture.json", architecture)
architecture

{'saved': 'output/course-labs/day2-v2/student-run/01_architecture.json'}


{'user_request': '회의를 이해하고 근거 있는 기록·할 일·인사이트 초안을 만들어 줘',
 'layers': [{'order': 1,
   'name': 'policy',
   'question': '읽어도 되는 정보와 하면 안 되는 행동은?'},
  {'order': 2,
   'name': 'input_adapter',
   'question': 'Meet·ClovaNote·음성 중 어떤 한 입력인가?'},
  {'order': 3, 'name': 'domain_context', 'question': '산업 용어와 이전 결정은 무엇인가?'},
  {'order': 4, 'name': 'workflow', 'question': '정규화→STT→구조화→근거 검증 순서는?'},
  {'order': 5, 'name': 'human_review', 'question': '누가 승인·수정·거절하는가?'},
  {'order': 6, 'name': 'draft_export', 'question': 'MD·이메일 초안을 어디까지 만들 것인가?'}],
 'two_meanings_of_agent': {'user_view': '여러 단계를 알아서 이어 주는 서비스',
  'engineering_view': '상태·도구·정책·오류·승인을 가진 실행 시스템'},
 'fixed_graph': ['policy',
  'input_normalize',
  'stt_optional',
  'structure',
  'evidence',
  'human_review',
  'export_draft'],
 'invariants': {'source_count': 1,
  'human_review_required': True,
  'external_write': False,
  'run_all_network_calls': 0},
 'three_route_cases': {'single_llm': {'strategy': 'single_llm',
   'reason': 'ONE_OFF

## 2차시 · Input Route · STT

Google Meet 텍스트, ClovaNote TXT, 로컬 음성은 출발점만 다릅니다. 텍스트가 이미 있으면 STT를 건너뛰고, 음성만 있을 때 로컬 STT를 실행합니다. 한 요청에 입력을 섞지 않고 모두 `TranscriptEnvelope`로 바꾼 뒤 같은 Workflow에 넣습니다.

In [5]:
import os
from src.meeting_demo import parse_transcript
from scripts.day2_public_audio import load_catalog, resolve, select_source

meet_text = "\n".join([
    "[00:00] 민지: 오늘은 배송 지연 회의 기록 자동화 범위를 확정하겠습니다.",
    "[00:18] 준호: WISMO 문의를 우선 처리하고 환불 자동화는 보류하는 것이 좋겠습니다.",
    "[00:37] 서연: 제가 9월 2일까지 고객 안내 문구를 정리해 공유하겠습니다.",
    "[00:55] 민지: 최근 야근 부담이 있으니 이번 범위를 더 늘리지 않겠습니다.",
])
clova_text = "\n".join([
    "화자 1 00:00", "배송 지연 원인 분류를 1차 범위로 확정합니다.",
    "화자 2 00:24", "제가 9월 3일까지 분류 기준을 작성하겠습니다.",
    "화자 1 00:46", "운영팀 부담을 확인한 뒤 다음 범위를 결정하겠습니다.",
])
audio_path = ROOT / "data/meeting_sample_ko_12min.wav"

sources = {
    "google_meet_text": SourceInput(
        source_mode="google_meet_text", source_ref="meet://fixture/delivery-001",
        meet_transcript=meet_text,
        speaker_metadata={
            "민지": {"display_name": "민지", "role": "PM"},
            "준호": {"display_name": "준호", "role": "Engineer"},
            "서연": {"display_name": "서연", "role": "Operations"},
        },
        history_metadata={"prior_decisions": ["고객 자동 발송 금지"]},
    ),
    "clovanote_txt": SourceInput(
        source_mode="clovanote_txt", source_ref="clovanote-export-001.txt",
        clovanote_text=clova_text,
        speaker_metadata={
            "화자 1": {"display_name": "민지", "role": "PM"},
            "화자 2": {"display_name": "준호", "role": "Engineer"},
        },
    ),
    "audio_stt": SourceInput(
        source_mode="audio_stt", source_ref="synthetic-12min-audio",
        audio_path=str(audio_path),
    ),
}

def reviewed_fixture_stt(path):
    assert path.resolve() == audio_path.resolve()
    text = (ROOT / "data/meeting_sample_ko_12min.txt").read_text(encoding="utf-8")
    segments = parse_transcript(text)
    return text, segments, {
        "provider": "reviewed_transcript_fixture", "language": "ko",
        "lane": "deterministic_run_all_not_live_stt",
        "review_status": "instructor_reviewed_audio_transcript_pair",
        "network_used": False, "matched_audio_transcript_pair": True,
    }

envelopes = {
    "google_meet_text": normalize_source(sources["google_meet_text"]),
    "clovanote_txt": normalize_source(sources["clovanote_txt"]),
    "audio_stt": normalize_source(
        sources["audio_stt"], transcriber=reviewed_fixture_stt
    ),
}
public_audio_catalog = load_catalog()
public_audio_source = select_source(public_audio_catalog, None)
public_audio_resolution = resolve(public_audio_catalog, public_audio_source)
resolved_public_audio = ROOT / public_audio_resolution["path"]
faster_whisper_live_opt_in = (
    os.getenv("FASTER_WHISPER_LIVE_OPT_IN", "0") == "1"
)
local_stt_smoke = run_optional_local_stt_smoke(
    resolved_public_audio,
    workspace_root=ROOT,
    live_opt_in=faster_whisper_live_opt_in,
    model_size=os.getenv("FASTER_WHISPER_MODEL", "small"),
)
input_result = {
    "reviewed_fixture_lane": {
        "label": "검토 완료 Transcript Fixture · 기본 Run All",
        "audio_path": str(audio_path.relative_to(ROOT)),
        "transcript_path": "data/meeting_sample_ko_12min.txt",
        "live_stt": False,
        "silent_transcript_substitution": False,
    },
    "optional_local_faster_whisper_smoke": {
        "opt_in_environment": "FASTER_WHISPER_LIVE_OPT_IN=1",
        "resolved_public_audio": public_audio_resolution,
        "result": local_stt_smoke,
    },
    "contracts": {
        name: {
            "source_mode": envelope.source_mode,
            "source_count": envelope.source_count,
            "segment_count": len(envelope.segments),
            "first_segment": envelope.segments[0].model_dump(mode="json"),
            "stt_metadata": envelope.stt_metadata,
        }
        for name, envelope in envelopes.items()
    },
    "boundary": source_mixing_error_example(),
}
assert {item["source_count"] for item in input_result["contracts"].values()} == {1}
assert input_result["boundary"]["error_code"] == "SOURCE_MODE_MIXING_FORBIDDEN"
assert envelopes["audio_stt"].stt_metadata["provider"] == "reviewed_transcript_fixture"
assert local_stt_smoke["transcript_substituted"] is False
if not faster_whisper_live_opt_in:
    assert local_stt_smoke["error_code"] == "LOCAL_STT_LIVE_OPT_IN_REQUIRED"
save_json("02_inputs.json", input_result)
input_result

{'saved': 'output/course-labs/day2-v2/student-run/02_inputs.json'}


{'reviewed_fixture_lane': {'label': '검토 완료 Transcript Fixture · 기본 Run All',
  'audio_path': 'data/meeting_sample_ko_12min.wav',
  'transcript_path': 'data/meeting_sample_ko_12min.txt',
  'live_stt': False,
  'silent_transcript_substitution': False},
 'optional_local_faster_whisper_smoke': {'opt_in_environment': 'FASTER_WHISPER_LIVE_OPT_IN=1',
  'resolved_public_audio': {'status': 'READY',
   'path': 'data/day2_public_audio/meeting_ko_ccby_excerpt_10m.mp3',
   'reason': 'CC_BY_EXCERPT_AVAILABLE',
   'provenance': 'public_cc_by_4_0',
   'automatic_external_write': False},
  'result': {'audio_path': 'data/day2_public_audio/meeting_ko_ccby_excerpt_10m.mp3',
   'source_mode': 'audio_stt',
   'transcript_substituted': False,
   'external_write': False,
   'status': 'EXPECTED_SKIP',
   'error_code': 'LOCAL_STT_LIVE_OPT_IN_REQUIRED',
   'live_attempted': False}},
 'contracts': {'google_meet_text': {'source_mode': 'google_meet_text',
   'source_count': 1,
   'segment_count': 4,
   'first_segme

## 3차시 · Domain Context · MCP Policy

회의에서 말한 사실과 사용자가 제공한 업무 맥락을 분리합니다. Notion·Confluence·Slack이 필요해 보여도 자동으로 읽지 않고, 허용된 범위와 기간을 가진 MCP 읽기 계획만 먼저 만듭니다.

In [6]:
domain_context = DomainContext(
    industry="이커머스 고객경험",
    organization_context="배송 지연 문의가 증가해 상담 부담과 고객 불편이 함께 커진 상태",
    meeting_objective="배송 지연 회의 기록 자동화 범위 확정",
    glossary={"WISMO": "배송 위치 문의", "SLA": "약속한 응답 시간"},
    prior_decisions=["외부 발송은 사람 승인 뒤에만 진행", "환불 자동화는 이번 범위에서 제외"],
    desired_outcomes=["근거가 있는 담당자별 To Do", "단기·중기·장기 인사이트"],
    confidentiality="internal",
)
retrieval_policy = MCPRetrievalPolicy(
    allowed_connectors=["notion", "confluence", "slack"],
    explicit_user_authorization=True,
    lookback_days=14,
    allowed_scopes={
        "notion": ["CX PoC"], "confluence": ["CX 정책"], "slack": ["#delivery-poc"],
    },
    participant_match_required=True,
    max_items_per_connector=5,
)
mcp_plan = build_mcp_retrieval_plan(
    envelope=envelopes["google_meet_text"],
    domain=domain_context,
    policy=retrieval_policy,
)
context_prompt_fields = {
    "industry": domain_context.industry,
    "organization_context": domain_context.organization_context,
    "meeting_objective": domain_context.meeting_objective,
    "glossary": domain_context.glossary,
    "prior_decisions": domain_context.prior_decisions,
    "desired_outputs": domain_context.desired_outcomes,
    "evidence_rule": "회의 발화의 사실 주장에는 s01 같은 evidence ID 필수",
}
context_result = {
    "domain_context": domain_context.model_dump(mode="json"),
    "context_prompt_fields": context_prompt_fields,
    "mcp_retrieval_plan": mcp_plan,
}
assert mcp_plan["executed"] is False and mcp_plan["external_write"] is False
save_json("03_domain_context.json", context_result)
context_result

{'saved': 'output/course-labs/day2-v2/student-run/03_domain_context.json'}


{'domain_context': {'industry': '이커머스 고객경험',
  'organization_context': '배송 지연 문의가 증가해 상담 부담과 고객 불편이 함께 커진 상태',
  'meeting_objective': '배송 지연 회의 기록 자동화 범위 확정',
  'glossary': {'WISMO': '배송 위치 문의', 'SLA': '약속한 응답 시간'},
  'prior_decisions': ['외부 발송은 사람 승인 뒤에만 진행', '환불 자동화는 이번 범위에서 제외'],
  'desired_outcomes': ['근거가 있는 담당자별 To Do', '단기·중기·장기 인사이트'],
  'confidentiality': 'internal'},
 'context_prompt_fields': {'industry': '이커머스 고객경험',
  'organization_context': '배송 지연 문의가 증가해 상담 부담과 고객 불편이 함께 커진 상태',
  'meeting_objective': '배송 지연 회의 기록 자동화 범위 확정',
  'glossary': {'WISMO': '배송 위치 문의', 'SLA': '약속한 응답 시간'},
  'prior_decisions': ['외부 발송은 사람 승인 뒤에만 진행', '환불 자동화는 이번 범위에서 제외'],
  'desired_outputs': ['근거가 있는 담당자별 To Do', '단기·중기·장기 인사이트'],
  'evidence_rule': '회의 발화의 사실 주장에는 s01 같은 evidence ID 필수'},
 'mcp_retrieval_plan': {'status': 'SIMULATED_POLICY_PLAN',
  'meeting_source_mode': 'google_meet_text',
  'operations': [{'connector': 'notion',
    'operation': 'search_read_only',
    'query_terms': ['배송 지연

## 4차시 · MeetingRecord Schema

먼저 모든 실행 방식이 반환해야 할 `MeetingRecord`를 고정합니다. 그 다음 한 번의 생성이면 단일 LLM, 고정 순서면 Workflow, 정보원과 다음 행동이 요청마다 달라지면 Agent Router를 선택합니다. 알려진 경로를 고르는 데 LLM을 쓰지 않으면 비용과 오작동 지점을 줄일 수 있습니다.

In [7]:
from pydantic import ValidationError

record_contract_run = run_meeting_workflow(
    sources["google_meet_text"],
    domain_context,
    review_decision="approve",
    retrieval_policy=retrieval_policy,
)
actual_record = MeetingRecord.model_validate(record_contract_run["record"])
actual_envelope = TranscriptEnvelope.model_validate(record_contract_run["envelope"])
actual_schema = MeetingRecord.model_json_schema()
routing_cases = {
    "fixed_meeting_record": route_execution_strategy(
        requested_actions=["normalize", "summarize", "perspectives", "todos", "insights", "draft"]
    ),
    "context_retrieval_needed": route_execution_strategy(
        requested_actions=["summarize", "todos"],
        external_context_sources=["notion", "confluence", "slack"],
    ),
    "one_off_unmodeled_request": route_execution_strategy(
        requested_actions=["rewrite_as_podcast_script"]
    ),
}
unknown_evidence_payload = actual_record.model_dump(mode="json")
unknown_evidence_payload["todos"][0]["evidence_ids"] = ["s999"]
unknown_evidence_errors = validate_record_evidence(
    MeetingRecord.model_validate(unknown_evidence_payload), actual_envelope
)

owner_due_run = run_meeting_workflow(
    SourceInput(
        source_mode="google_meet_text",
        source_ref="meet://boundary/owner-due-unknown",
        meet_transcript="민지: 다음 회의에서 후속 조치를 논의해 봅시다.",
    ),
    domain_context,
    review_decision="approve",
)
owner_due_todo = owner_due_run["record"]["todos"][0]

invalid_due_payload = actual_record.model_dump(mode="json")
invalid_due_payload["todos"][0]["due_date"] = "2026-02-30"
try:
    MeetingRecord.model_validate(invalid_due_payload)
    invalid_due_boundary = {"status": "UNEXPECTED_SUCCESS"}
except ValidationError:
    invalid_due_boundary = {
        "status": "EXPECTED_FAILURE",
        "error_code": "TODO_DUE_DATE_INVALID",
    }

additional_field_payload = actual_record.model_dump(mode="json")
additional_field_payload["automatic_send"] = True
try:
    MeetingRecord.model_validate(additional_field_payload)
    additional_field_boundary = {"status": "UNEXPECTED_SUCCESS"}
except ValidationError:
    additional_field_boundary = {
        "status": "EXPECTED_FAILURE",
        "error_code": "MEETING_RECORD_ADDITIONAL_FIELD_FORBIDDEN",
    }
record_contract_result = {
    "schema": {
        "name": "MeetingRecord",
        "field_names": list(MeetingRecord.model_fields),
        "required_fields": actual_schema.get("required", []),
        "properties": actual_schema["properties"],
    },
    "actual_record": actual_record.model_dump(mode="json"),
    "evidence_validation": {
        "known_segment_ids": [segment.id for segment in actual_envelope.segments],
        "errors": validate_record_evidence(actual_record, actual_envelope),
    },
    "execution_strategy_comparison": compare_execution_strategies(),
    "rule_router_examples": routing_cases,
    "boundary_evidence": {
        "unknown_evidence_s999": unknown_evidence_errors,
        "owner_due_not_in_source": {
            "owner": owner_due_todo["owner"],
            "due_date": owner_due_todo["due_date"],
            "rule": "원문에 없으면 null 유지",
        },
        "invalid_due_date": invalid_due_boundary,
        "additional_field": additional_field_boundary,
    },
    "delivery_policy": {
        "human_review_required": actual_record.human_review_required,
        "external_write": actual_record.external_write,
        "draft_status": record_contract_run["exports"]["status"],
    },
}
assert set(record_contract_result["actual_record"]) == set(MeetingRecord.model_fields)
assert record_contract_result["evidence_validation"]["errors"] == []
assert record_contract_result["delivery_policy"]["external_write"] is False
assert unknown_evidence_errors == ["TODO_1_UNKNOWN_EVIDENCE:s999"]
assert owner_due_todo["owner"] is None and owner_due_todo["due_date"] is None
assert invalid_due_boundary["error_code"] == "TODO_DUE_DATE_INVALID"
assert additional_field_boundary["error_code"] == (
    "MEETING_RECORD_ADDITIONAL_FIELD_FORBIDDEN"
)
assert routing_cases["fixed_meeting_record"]["strategy"] == "deterministic_workflow"
assert routing_cases["context_retrieval_needed"]["strategy"] == "agent_router"
assert routing_cases["one_off_unmodeled_request"]["strategy"] == "single_llm"
save_json("04_meeting_record_contract.json", record_contract_result)
record_contract_result

{'saved': 'output/course-labs/day2-v2/student-run/04_meeting_record_contract.json'}


{'schema': {'name': 'MeetingRecord',
  'field_names': ['title',
   'domain',
   'purpose',
   'previous_context',
   'meeting_summary',
   'summary_evidence_ids',
   'participant_perspectives',
   'todos',
   'insights',
   'wellbeing_risks',
   'evidence_segment_count',
   'status',
   'human_review_required',
   'external_write'],
  'required_fields': ['title',
   'domain',
   'purpose',
   'previous_context',
   'meeting_summary',
   'summary_evidence_ids',
   'participant_perspectives',
   'todos',
   'insights',
   'evidence_segment_count'],
  'properties': {'title': {'minLength': 1, 'title': 'Title', 'type': 'string'},
   'domain': {'minLength': 1, 'title': 'Domain', 'type': 'string'},
   'purpose': {'minLength': 1, 'title': 'Purpose', 'type': 'string'},
   'previous_context': {'minLength': 1,
    'title': 'Previous Context',
    'type': 'string'},
   'meeting_summary': {'minLength': 1,
    'title': 'Meeting Summary',
    'type': 'string'},
   'summary_evidence_ids': {'items': {'

## 5차시 · Coding Agent Workflow

Codex나 Claude Code에는 곧바로 “Agent를 만들어 줘”라고 하지 않습니다. 세 입력 시나리오, 공통 결과 계약, 금지 행동, 정상·실패 Test를 먼저 합의한 뒤 구현을 요청합니다. 아래 실행 결과가 대화형 코딩 Agent에게 전달할 인수 기준입니다.

In [8]:
coding_agent_brief = {
    "scenarios": ["google_meet_text", "clovanote_txt", "audio_stt"],
    "implementation_request": (
        "세 입력을 하나의 MeetingRecord로 정규화하고, 근거 검증과 사람 승인 뒤 "
        "Markdown·이메일 초안까지만 만드는 코드를 구현해 주세요."
    ),
    "must_not": ["입력 자동 혼합", "근거 없는 담당자 추정", "승인 전 외부 저장·발송"],
    "acceptance_tests": [
        "세 입력 모두 같은 결과 계약", "존재하지 않는 evidence ID 차단",
        "승인·수정·거절 상태 분리", "모든 기본 실행에서 external_write=false",
    ],
    "conversation_guide": "materials/day2/Codex_Claude_대화_시나리오.md",
}
workflow_runs = {
    "google_meet_text": run_meeting_workflow(
        sources["google_meet_text"], domain_context,
        review_decision="approve", retrieval_policy=retrieval_policy,
    ),
    "clovanote_txt": run_meeting_workflow(
        sources["clovanote_txt"], domain_context,
        review_decision="approve",
    ),
    "audio_stt": run_meeting_workflow(
        sources["audio_stt"], domain_context,
        review_decision="edit",
        review_edits={
            "meeting_summary": "고객 문의 자동화 PoC의 범위·금지 행동·담당자별 후속 조치를 근거와 함께 정리했습니다."
        },
        transcriber=reviewed_fixture_stt,
    ),
}
workflow_result = {
    "coding_agent_brief": coding_agent_brief,
    "scenarios": {
        name: compact_workflow_result(result)
        for name, result in workflow_runs.items()
    },
    "full_records": {
        name: result["record"] for name, result in workflow_runs.items()
    },
}
expected_nodes = [
    "policy", "input_normalize", "stt_optional", "structure",
    "evidence", "human_review", "export_draft",
]
assert all(
    [event["node"] for event in item["trace"]] == expected_nodes
    for item in workflow_result["scenarios"].values()
)
assert all(item["status"] == "DRAFT_READY" for item in workflow_result["scenarios"].values())
assert all(item["external_write"] is False for item in workflow_result["scenarios"].values())
save_json("05_workflow_runs.json", workflow_result)
workflow_result["scenarios"]

{'saved': 'output/course-labs/day2-v2/student-run/05_workflow_runs.json'}


{'google_meet_text': {'status': 'DRAFT_READY',
  'source_mode': 'google_meet_text',
  'segment_count': 4,
  'purpose': '배송 지연 회의 기록 자동화 범위 확정',
  'summary': '오늘은 배송 지연 회의 기록 자동화 범위를 확정하겠습니다. WISMO 문의를 우선 처리하고 환불 자동화는 보류하는 것이 좋겠습니다.',
  'participant_count': 3,
  'todo_count': 1,
  'wellbeing_risk_count': 1,
  'review': {'decision': 'approve',
   'status': 'APPROVED_READY_FOR_DRAFT',
   'export_ready': True,
   'human_reviewed': True,
   'external_write': False},
  'trace': [{'node': 'policy',
    'status': 'SUCCESS',
    'external_write': False,
    'source_mode': 'google_meet_text',
    'source_count': 1,
    'human_review_required': True},
   {'node': 'input_normalize',
    'status': 'SUCCESS',
    'external_write': False,
    'source_mode': 'google_meet_text',
    'segment_count': 4},
   {'node': 'stt_optional',
    'status': 'SKIPPED_TEXT_INPUT',
    'external_write': False},
   {'node': 'structure',
    'status': 'SUCCESS',
    'external_write': False,
    'participant_count': 3,
 

## 6차시 · LLM Provider · Cost Guardrail

기본 `Run All`은 API와 CLI를 호출하지 않습니다. `.env`를 읽되 OpenAI는 `OPENAI_LIVE_OPT_IN=1`, Ollama는 `OLLAMA_LIVE_OPT_IN=1`일 때만 실행합니다. OpenAI 모델 접근 불가를 fixture 성공으로 위장하지 않고, Ollama 출력도 Schema와 evidence를 통과해야 성공입니다.

In [9]:
from types import SimpleNamespace
from dotenv import load_dotenv

dotenv_loaded = load_dotenv(dotenv_path=ROOT / ".env", override=False)
openai_live_opt_in = os.getenv("OPENAI_LIVE_OPT_IN", "0") == "1"
ollama_live_opt_in = os.getenv("OLLAMA_LIVE_OPT_IN", "0") == "1"

provider_options = diagnose_provider_options()
cli_dry_runs = {
    name: run_optional_cli_prompt(name, "현재 회의 기록을 검토해 주세요.")
    for name in ("ollama", "codex", "claude_code")
}
openai_result = run_optional_openai_record(
    envelopes["google_meet_text"], domain_context,
    env=os.environ if openai_live_opt_in else {},
    model=os.getenv("OPENAI_MODEL", DEFAULT_OPENAI_MODEL),
    allow_fixture_fallback=True,
)

ollama_prompt = "\n".join([
    "다음 MeetingRecord JSON Schema에 맞는 JSON object만 반환하세요.",
    "원문에 없는 owner와 due_date는 null, evidence ID는 제공된 값만 사용하세요.",
    "evidence_ids의 각 값은 ALLOWED_EVIDENCE_IDS의 문자열과 정확히 같아야 합니다.",
    "human_review_required=true, external_write=false를 유지하세요.",
    "ALLOWED_EVIDENCE_IDS=" + json.dumps(
        [item.id for item in envelopes["google_meet_text"].segments],
        ensure_ascii=False,
    ),
    "SCHEMA=" + json.dumps(MeetingRecord.model_json_schema(), ensure_ascii=False),
    "TRANSCRIPT=" + envelopes["google_meet_text"].transcript_text,
])
ollama_call = run_optional_cli_prompt(
    "ollama",
    ollama_prompt,
    live_opt_in=ollama_live_opt_in,
    model=os.getenv("OLLAMA_MODEL", "qwen3:4b"),
)
if ollama_call["status"] == "SUCCESS":
    ollama_validation = {
        "provider_status": "SUCCESS",
        **validate_model_record_output(
            ollama_call["output_text"], envelopes["google_meet_text"]
        ),
    }
else:
    ollama_validation = {
        "status": ollama_call["status"],
        "provider_status": ollama_call["status"],
        "error_code": (
            "OLLAMA_LIVE_OPT_IN_REQUIRED"
            if ollama_call["error_code"] == "CLI_LIVE_OPT_IN_REQUIRED"
            else ollama_call["error_code"]
        ),
        "command_executed": ollama_call["command_executed"],
        "schema_valid": False,
        "evidence_valid": False,
        "fallback_used": False,
        "external_write": False,
    }

class LocalModelNotFound(Exception):
    status_code = 404

class FakeResponses:
    @staticmethod
    def create(**_kwargs):
        raise LocalModelNotFound("requested model does not exist")

model_boundary = run_optional_openai_prompt(
    "모델 가용성 경계 테스트",
    env={"OPENAI_LIVE_OPT_IN": "1"},
    client=SimpleNamespace(responses=FakeResponses()),
    model=DEFAULT_OPENAI_MODEL,
)
provider_result = {
    "options": provider_options,
    "cli_default_run_all": cli_dry_runs,
    "openai_default_run_all": openai_result,
    "ollama_optional_schema_evidence_validation": ollama_validation,
    "model_not_available_boundary": model_boundary,
    "dotenv": {
        "loaded_without_override": bool(dotenv_loaded),
        "credential_values_exposed": False,
    },
    "live_flags": {
        "openai": openai_live_opt_in,
        "ollama": ollama_live_opt_in,
        "faster_whisper": faster_whisper_live_opt_in,
        "codex_cli": False, "claude_code_cli": False,
    },
}
if not openai_live_opt_in:
    assert openai_result["provider_used"] == "fixture"
    assert openai_result["fallback_reason"] == "OPENAI_LIVE_OPT_IN_REQUIRED"
    assert openai_result["schema_valid"] is True
if not ollama_live_opt_in:
    assert ollama_validation["error_code"] == "OLLAMA_LIVE_OPT_IN_REQUIRED"
    assert ollama_validation["command_executed"] is False
assert model_boundary["fallback_reason"] == "MODEL_NOT_AVAILABLE"
assert all(item["error_code"] == "CLI_LIVE_OPT_IN_REQUIRED" for item in cli_dry_runs.values())
assert all(
    item.get("command_executed", False) is False
    for item in provider_options.values()
    if isinstance(item, dict)
)
save_json("06_provider_diagnostics.json", provider_result)
provider_result

{'saved': 'output/course-labs/day2-v2/student-run/06_provider_diagnostics.json'}


{'options': {'ollama': {'command': 'ollama',
   'opt_in_example': 'ollama run qwen3:4b',
   'role': '로컬 LLM 선택 실습',
   'installed': True,
   'status': 'INSTALLED_NOT_EXECUTED',
   'auth_checked': False,
   'command_executed': False,
   'credential_value_read': False,
   'external_write': False},
  'codex': {'command': 'codex',
   'opt_in_example': "codex exec --ephemeral --sandbox read-only '<요청>'",
   'role': '저장소 분석·코드 생성 Harness',
   'installed': True,
   'status': 'INSTALLED_NOT_EXECUTED',
   'auth_checked': False,
   'command_executed': False,
   'credential_value_read': False,
   'external_write': False},
  'claude_code': {'command': 'claude',
   'opt_in_example': "claude -p --tools '' --no-session-persistence '<요청>'",
   'role': '대체 코딩 Agent Harness',
   'installed': True,
   'status': 'INSTALLED_NOT_EXECUTED',
   'auth_checked': False,
   'command_executed': False,
   'credential_value_read': False,
   'external_write': False},
  'openai_api': {'default_model': 'gpt-5.6-luna',


## 7차시 · LangGraph · Human Review

먼저 `interrupt()`에서 멈춘 상태를 확인하고, 수강생이 결정값을 정한 뒤 같은 thread를 `Command(resume=...)`로 재개합니다. 마지막 자동 회귀 검증은 승인·수정·거절 세 경로가 계속 안전한지 별도로 확인합니다.

In [10]:
# 1단계 · interrupt 시작: 아직 승인 결정도, 초안 export도 없습니다.
learner_graph = build_interruptible_meeting_graph()
learner_thread_id = "day2-learner-review"
learner_start = start_interruptible_meeting_review(
    learner_graph,
    sources["google_meet_text"],
    domain_context,
    thread_id=learner_thread_id,
    retrieval_policy=retrieval_policy,
)
assert learner_start["status"] == "WAITING_FOR_HUMAN_REVIEW"
assert "exports" not in learner_start

# 2단계 · 수강생 결정/재개: 아래 두 값을 바꾼 뒤 이 셀을 다시 실행합니다.
LEARNER_REVIEW_DECISION = "edit"  # approve | edit | reject
LEARNER_REVIEW_EDITS = {
    "meeting_summary": "사람이 근거를 확인하고 배송 지연 기록 범위를 수정했습니다.",
    "todo_updates": {"0": {"owner": "민지", "due_date": "2026-09-05"}},
}
learner_resume = resume_interruptible_meeting_review(
    learner_graph,
    thread_id=learner_thread_id,
    decision=LEARNER_REVIEW_DECISION,
    edits=LEARNER_REVIEW_EDITS if LEARNER_REVIEW_DECISION == "edit" else {},
)
assert learner_resume["external_write"] is False

# 3단계 · 자동 회귀 evidence: 학습자 선택과 별도로 세 경로를 모두 검사합니다.
review_inputs = {
    "approve": {},
    "edit": {
        "meeting_summary": "사람이 근거를 확인하고 배송 지연 기록 범위를 수정했습니다.",
        "todo_updates": {"0": {"owner": "민지", "due_date": "2026-09-05"}},
    },
    "reject": {},
}
regression_runs = {}
for decision, edits in review_inputs.items():
    regression_graph = build_interruptible_meeting_graph()
    regression_thread_id = f"day2-regression-{decision}"
    regression_start = start_interruptible_meeting_review(
        regression_graph,
        sources["google_meet_text"],
        domain_context,
        thread_id=regression_thread_id,
        retrieval_policy=retrieval_policy,
    )
    regression_resume = resume_interruptible_meeting_review(
        regression_graph,
        thread_id=regression_thread_id,
        decision=decision,
        edits=edits,
    )
    regression_runs[decision] = {
        "start": {
            "status": regression_start["status"],
            "thread_id": regression_start["thread_id"],
            "checkpointer": regression_start["checkpointer"],
            "interrupt": regression_start["interrupts"][0],
            "external_write": regression_start["external_write"],
        },
        "resume": {
            "status": regression_resume["status"],
            "review": regression_resume["review"],
            "export_status": regression_resume["exports"]["status"],
            "trace": regression_resume["trace"],
            "external_write": regression_resume["external_write"],
        },
    }

human_review_result = {
    "graph": {
        "framework": "LangGraph",
        "checkpointer": "InMemorySaver",
        "pause": "interrupt()",
        "resume": "Command(resume=...)",
        "conditional_routes": [
            "evidence → human_review | evidence_hold",
            "human_review → export_draft | review_rejected",
        ],
    },
    "learner_interrupt_start": {
        "status": learner_start["status"],
        "thread_id": learner_start["thread_id"],
        "interrupt": learner_start["interrupts"][0],
        "external_write": learner_start["external_write"],
    },
    "learner_decision_resume": {
        "decision": LEARNER_REVIEW_DECISION,
        "status": learner_resume["status"],
        "review": learner_resume["review"],
        "export_status": learner_resume["exports"]["status"],
        "external_write": learner_resume["external_write"],
    },
    "automated_regression_evidence": regression_runs,
    "unknown_evidence_boundary_from_4th_period": (
        record_contract_result["boundary_evidence"]["unknown_evidence_s999"]
    ),
}
assert all(
    item["start"]["status"] == "WAITING_FOR_HUMAN_REVIEW"
    for item in regression_runs.values()
)
assert regression_runs["approve"]["resume"]["status"] == "DRAFT_READY"
assert regression_runs["edit"]["resume"]["status"] == "DRAFT_READY"
assert regression_runs["reject"]["resume"]["status"] == "REJECTED"
assert regression_runs["reject"]["resume"]["export_status"] == "SKIPPED_NOT_APPROVED"
assert all(
    item["resume"]["external_write"] is False
    for item in regression_runs.values()
)
assert human_review_result["unknown_evidence_boundary_from_4th_period"] == [
    "TODO_1_UNKNOWN_EVIDENCE:s999"
]
save_json("07_human_review.json", human_review_result)
human_review_result

{'saved': 'output/course-labs/day2-v2/student-run/07_human_review.json'}


{'graph': {'framework': 'LangGraph',
  'checkpointer': 'InMemorySaver',
  'pause': 'interrupt()',
  'resume': 'Command(resume=...)',
  'conditional_routes': ['evidence → human_review | evidence_hold',
   'human_review → export_draft | review_rejected']},
 'learner_interrupt_start': {'status': 'WAITING_FOR_HUMAN_REVIEW',
  'thread_id': 'day2-learner-review',
  'interrupt': {'id': '61cbc4bf638914e8057bae82dc75c742',
   'value': {'kind': 'MEETING_RECORD_HUMAN_REVIEW',
    'thread_id': 'day2-learner-review',
    'status': 'WAITING_FOR_HUMAN_REVIEW',
    'allowed_decisions': ['approve', 'edit', 'reject'],
    'record': {'title': '배송 지연 회의 기록 자동화 범위 확정 회의 기록',
     'domain': '이커머스 고객경험',
     'purpose': '배송 지연 회의 기록 자동화 범위 확정',
     'previous_context': '외부 발송은 사람 승인 뒤에만 진행; 환불 자동화는 이번 범위에서 제외',
     'meeting_summary': '오늘은 배송 지연 회의 기록 자동화 범위를 확정하겠습니다. WISMO 문의를 우선 처리하고 환불 자동화는 보류하는 것이 좋겠습니다.',
     'summary_evidence_ids': ['s01', 's02'],
     'participant_perspectives': [{'participant': '민지'

## 8차시 · Localhost App · 선택 Package

세 시나리오의 결과를 로컬 Markdown과 이메일 초안으로 만들고 같은 핵심 기능을 localhost UI에서 실제 실행합니다. 수강생 기본 경로는 Python 환경을 재사용하는 localhost이며, unsigned EXE·PKG는 Docker가 준비된 강사 PC의 선택 시연입니다.

In [11]:
markdown_files = {}
email_drafts = {}
for name, result in workflow_runs.items():
    record = MeetingRecord.model_validate(result["record"])
    markdown_text = result["exports"]["markdown"]
    markdown_path = save_text(f"08_{name}_meeting.md", markdown_text)
    markdown_files[name] = str(markdown_path.relative_to(ROOT))
    email_drafts[name] = render_email_draft(record, audience="internal")

save_json("08_email_drafts.json", email_drafts)
localhost_report_path = OUT / "08_localhost_launch.json"
localhost_smoke = run_command(
    sys.executable,
    "scripts/run_day2_local_app.py",
    "--smoke-and-exit",
    "--port",
    "0",
    "--report",
    str(localhost_report_path.relative_to(ROOT)),
)
assert localhost_smoke["returncode"] == 0
localhost_report = json.loads(localhost_report_path.read_text(encoding="utf-8"))
save_json("08_localhost_launch.json", localhost_report)
desktop_delivery = {
    "primary_run": "python scripts/run_day2_local_app.py",
    "macos_double_click": "desktop-app/meeting-intelligence/scripts/run-local.command",
    "windows_double_click": "desktop-app\\meeting-intelligence\\scripts\\run-local.cmd",
    "port_recovery": "python scripts/run_day2_local_app.py --port 0",
    "minimum_install": "python -m pip install -r desktop-app/meeting-intelligence/requirements-localhost.txt",
    "browser": "http://127.0.0.1:8766",
    "optional_package": {
        "windows_exe": "desktop-app/meeting-intelligence/dist/MeetingIntelligence-Windows.exe",
        "macos_pkg": "desktop-app/meeting-intelligence/dist/MeetingIntelligence-macOS.pkg",
        "docker_required": True,
        "signed": False,
        "role": "instructor_optional_demo",
    },
    "human_review_required": True,
    "external_write": False,
}
focused_test = run_command(
    sys.executable, "-m", "pytest", "-q", "tests/test_day2_meeting_workflow.py"
)
day1_suite = run_command(
    sys.executable, "-m", "pytest", "-q",
    "tests/test_day1_agent.py",
    "tests/test_langchain_langgraph_lab.py",
    "tests/test_meeting_agent_workflow.py",
    "tests/test_openai_provider.py",
    "tests/test_ollama_tool_agent.py",
)
focused_test_evidence = record_test_evidence("day2_focused", focused_test)
day1_test_evidence = record_test_evidence("day1_regression", day1_suite)
localhost_test_evidence = record_test_evidence("localhost_http_smoke", localhost_smoke)
export_result = {
    "markdown_files": markdown_files,
    "email_drafts": email_drafts,
    "desktop_delivery": desktop_delivery,
    "localhost_launch": localhost_report,
    "test_evidence": [focused_test_evidence, day1_test_evidence, localhost_test_evidence],
    "checks": {
        "all_emails_unsent": all(item["send"] is False for item in email_drafts.values()),
        "all_external_write_false": all(item["external_write"] is False for item in email_drafts.values()),
        "focused_test_returncode": focused_test["returncode"],
        "day1_suite_returncode": day1_suite["returncode"],
        "localhost_smoke_returncode": localhost_smoke["returncode"],
        "localhost_smoke_status": localhost_report["status"],
        "localhost_external_write_false": localhost_report["external_write"] is False,
    },
}
assert export_result["checks"] == {
    "all_emails_unsent": True,
    "all_external_write_false": True,
    "focused_test_returncode": 0,
    "day1_suite_returncode": 0,
    "localhost_smoke_returncode": 0,
    "localhost_smoke_status": "PASS",
    "localhost_external_write_false": True,
}
save_json("08_export_drafts.json", export_result)
export_result

{'saved': 'output/course-labs/day2-v2/student-run/08_google_meet_text_meeting.md'}
{'saved': 'output/course-labs/day2-v2/student-run/08_clovanote_txt_meeting.md'}
{'saved': 'output/course-labs/day2-v2/student-run/08_audio_stt_meeting.md'}
{'saved': 'output/course-labs/day2-v2/student-run/08_email_drafts.json'}


{
  "command": "python scripts/run_day2_local_app.py --smoke-and-exit --port 0 --report output/course-labs/day2-v2/student-run/08_localhost_launch.json",
  "returncode": 0,
  "stdout_tail": [
    "{\"status\": \"PASS\", \"url\": \"http://127.0.0.1:58982\", \"report\": \"/Users/sungjae-cha/sungjae-cha/llm-agent-and-workflow-automation/output/course-labs/day2-v2/student-run/08_localhost_launch.json\"}"
  ],
  "stderr_tail": []
}
{'saved': 'output/course-labs/day2-v2/student-run/08_localhost_launch.json'}


{
  "command": "python -m pytest -q tests/test_day2_meeting_workflow.py",
  "returncode": 0,
  "stdout_tail": [
    "..................                                                       [100%]",
    "18 passed in 0.20s"
  ],
  "stderr_tail": []
}


{
  "command": "python -m pytest -q tests/test_day1_agent.py tests/test_langchain_langgraph_lab.py tests/test_meeting_agent_workflow.py tests/test_openai_provider.py tests/test_ollama_tool_agent.py",
  "returncode": 0,
  "stdout_tail": [
    "...................................                                      [100%]",
    "35 passed in 0.44s"
  ],
  "stderr_tail": []
}
{'saved': 'output/course-labs/day2-v2/student-run/08_export_drafts.json'}


{'markdown_files': {'google_meet_text': 'output/course-labs/day2-v2/student-run/08_google_meet_text_meeting.md',
  'clovanote_txt': 'output/course-labs/day2-v2/student-run/08_clovanote_txt_meeting.md',
  'audio_stt': 'output/course-labs/day2-v2/student-run/08_audio_stt_meeting.md'},
 'email_drafts': {'google_meet_text': {'subject': '[회의 기록] 배송 지연 회의 기록 자동화 범위 확정 회의 기록',
   'audience': 'internal',
   'to': [],
   'body': '안녕하세요, 회의 참석자 여러분.\n\n배송 지연 회의 기록 자동화 범위 확정 관련 회의 내용을 아래와 같이 정리했습니다.\n\n오늘은 배송 지연 회의 기록 자동화 범위를 확정하겠습니다. WISMO 문의를 우선 처리하고 환불 자동화는 보류하는 것이 좋겠습니다.\n\n[후속 조치]\n- 제가 9월 2일까지 고객 안내 문구를 정리해 공유하겠습니다. / 서연 / 2026-09-02\n\n담당자·기한·대외 공유 범위를 확인한 뒤 발송해 주세요.',
   'send': False,
   'external_write': False,
   'human_recipient_check_required': True},
  'clovanote_txt': {'subject': '[회의 기록] 배송 지연 회의 기록 자동화 범위 확정 회의 기록',
   'audience': 'internal',
   'to': [],
   'body': '안녕하세요, 회의 참석자 여러분.\n\n배송 지연 회의 기록 자동화 범위 확정 관련 회의 내용을 아래와 같이 정리했습니다.\n\n배송 지연 원인 분류를 1차 범위로 확정합니다. 운영팀 부담을 확인한 뒤 다

## 완료 확인

- Day 2의 1~8차시 결과 파일을 확인했습니다.
- 정상 경로와 가장 중요한 실패 경로를 모두 실행했습니다.
- 외부 서비스 저장·게시·발송과 자동 메일이 기본값 `false`임을 확인했습니다.
- Codex·Claude Code 결과는 test와 diff를 사람이 검토한 뒤에만 반영합니다.